In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

In [2]:
url="https://www.zameen.com/Houses_Property/Lahore-1-1.html"


In [3]:
response=requests.get(url)
soup=BeautifulSoup(response.content,'html.parser')

In [4]:
all_sites=soup.find(class_="_5d76b098")

In [5]:
len(all_sites.find_all("li",class_="_6d61972c"))
all_socities=all_sites.find_all("li",class_="_6d61972c")
all_socities

[<li class="_6d61972c"><a class="_6de4d4f7" href="/Houses_Property/Lahore_DHA_Defence-9-1.html">DHA Defence<span class="_4519bd96">(<!-- -->7,583<!-- -->)</span></a></li>,
 <li class="_6d61972c"><a class="_6de4d4f7" href="/Houses_Property/Lahore_Bahria_Town-509-1.html">Bahria Town<span class="_4519bd96">(<!-- -->1,680<!-- -->)</span></a></li>,
 <li class="_6d61972c"><a class="_6de4d4f7" href="/Houses_Property/Lahore_Park_View_City-1466-1.html">Park View City<span class="_4519bd96">(<!-- -->1,668<!-- -->)</span></a></li>,
 <li class="_6d61972c"><a class="_6de4d4f7" href="/Houses_Property/Lahore_Raiwind_Road-128-1.html">Raiwind Road<span class="_4519bd96">(<!-- -->883<!-- -->)</span></a></li>,
 <li class="_6d61972c"><a class="_6de4d4f7" href="/Houses_Property/Lahore_Johar_Town-93-1.html">Johar Town<span class="_4519bd96">(<!-- -->651<!-- -->)</span></a></li>,
 <li class="_6d61972c"><a class="_6de4d4f7" href="/Houses_Property/Lahore_GT_Road-472-1.html">GT Road<span class="_4519bd96">(<!--

In [6]:
base_url="https://www.zameen.com"

In [7]:
def scrape_property_links(listing_url):
    """Extract property links from a listing page"""
    try:
        response = requests.get(listing_url, timeout=20)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, "html.parser")

        ul_tag = soup.find("ul", class_="e20beb46")
        links = []
        if ul_tag:
            for a_tag in ul_tag.find_all("a", class_="d870ae17", href=True):
                href = a_tag["href"]
                if href.startswith("/"):
                    links.append(base_url + href)
                else:
                    links.append(href)
        return links
    except Exception as e:
        print(f"Error scraping listing page {listing_url}: {e}")
        return []

In [8]:
for society in all_socities:
    print(society.find("a").get("href"))
    link=society.find("a").get("href")
    link=base_url+link
    print(link)
    response=requests.get(link)
    soup=BeautifulSoup(response.content,'html.parser')
    
    list_of_houses=soup.find("ul",class_="e20beb46")
    print(list_of_houses)
    break
    

/Houses_Property/Lahore_DHA_Defence-9-1.html
https://www.zameen.com/Houses_Property/Lahore_DHA_Defence-9-1.html
<ul class="e20beb46"><li aria-label="Listing" class="a37d52f0" role="article"><article class="_5b98ebdf"><div class="_2a2e3d21"><a aria-label="Listing link" class="d870ae17" href="/Property/dha_defence_dha_9_town_5-marla_semi_furnished_corner_spanish_villa_near_park_for_sale_in_dha-52234706-1452-1.html" index="0" title="5-Marla Semi Furnished Corner Spanish Villa Near Park For Sale In DHA"><div class="_3547d663"></div></a></div><div class="_317a748d _2017e45b"><div class="_77d2975b"><picture class="a659dd2e"><source srcset="https://media.zameen.com/thumbnails/290963155-400x300.webp" type="image/webp"/><img alt="5-Marla Semi Furnished Corner Spanish Villa Near Park For Sale In DHA" aria-label="Listing photo" class="fd6eb59d" fetchpriority="low" loading="eager" role="presentation" src="https://media.zameen.com/thumbnails/290963155-400x300.jpeg" title="5-Marla Semi Furnished Cor

In [10]:
import re
import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE = "https://www.zameen.com"

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9"
})

data = []
all_feature_keys = set()
seen_property_ids = set()

# -----------------------
# Overview label helpers (same as your logic)
# -----------------------
def is_area_li(tag):
    if tag.name != "li":
        return False
    label = tag.find("span", class_="ed0db22a")
    return label and label.get_text(strip=True) == "Area"

def is_bedroom_li(tag):
    if tag.name != "li":
        return False
    label = tag.find("span", class_="ed0db22a")
    return label and label.get_text(strip=True) in ["Bedroom(s)", "Beds"]

def is_bath_li(tag):
    if tag.name != "li":
        return False
    label = tag.find("span", class_="ed0db22a")
    return label and label.get_text(strip=True) in ["Bath(s)", "Baths"]

def get_soup(url, timeout=25):
    r = session.get(url, timeout=timeout)
    r.raise_for_status()
    return BeautifulSoup(r.content, "html.parser")

def make_page_url(seed_url: str, page: int) -> str:
    """
    Convert:
      ...-1.html -> ...-2.html -> ...-3.html
    """
    if re.search(r"-\d+\.html$", seed_url):
        return re.sub(r"-\d+\.html$", f"-{page}.html", seed_url)
    # fallback
    return seed_url

# -----------------------
# 1) Extract societies from master page
# -----------------------
def extract_societies(master_url: str):
    soup = get_soup(master_url)
    wrapper = soup.find(class_="_5d76b098")   # your container
    if not wrapper:
        return []

    items = wrapper.find_all("li", class_="_6d61972c")
    societies = []

    for li in items:
        a = li.find("a", class_="_6de4d4f7", href=True)
        if not a:
            continue

        href = urljoin(BASE, a["href"])
        full_text = a.get_text(" ", strip=True)  # "DHA Defence (7,554)"

        # name = text without the trailing (count)
        name = re.sub(r"\(\s*[\d,]+\s*\)\s*$", "", full_text).strip()

        # count as int (optional)
        m = re.search(r"\(\s*([\d,]+)\s*\)\s*$", full_text)
        count = int(m.group(1).replace(",", "")) if m else None

        societies.append({
            "society_name": name,
            "society_count": count,
            "society_url": href
        })

    return societies

# -----------------------
# 2) Listing page -> property links
# -----------------------
def scrape_property_links(listing_url: str):
    soup = get_soup(listing_url)
    ul = soup.find("ul", class_="e20beb46")
    links = []
    if ul:
        for a in ul.find_all("a", class_="d870ae17", href=True):
            links.append(urljoin(BASE, a["href"]))
    return links

# -----------------------
# 3) Property page -> details (same features logic)
# -----------------------
def scrape_property_details(prop_url: str, society_name: str, society_url: str):
    global all_feature_keys

    soup = get_soup(prop_url)

    title_tag = soup.find(class_="cd230541")
    price_tag = soup.find(class_="_2923a568")

    area_li = soup.find(is_area_li)
    bed_li  = soup.find(is_bedroom_li)
    bath_li = soup.find(is_bath_li)

    # basic values
    bed_count = bath_count = marla_value = area_sqft = None

    if bed_li:
        s = bed_li.find("span", class_="_2fdf7fc5")
        bed_count = s.get_text(strip=True) if s else None

    if bath_li:
        s = bath_li.find("span", class_="_2fdf7fc5")
        bath_count = s.get_text(strip=True) if s else None

    if area_li:
        s = area_li.find("span", class_="_2fdf7fc5")
        if s:
            area_text = s.get_text(strip=True)
            try:
                marla_value = float(area_text.split()[0])
                area_sqft = marla_value * 225
            except:
                pass

    # price
    price_raw = price_tag.get_text(strip=True) if price_tag else ""
    price = "PKR " + price_raw[3:].strip() if (price_raw.startswith("PKR") and not price_raw.startswith("PKR ")) else price_raw

    # address/title
    page_title = title_tag.get_text(strip=True) if title_tag else "Not listed"

    # ✅ required naming format
    # e.g. "5 Bedroom House for sale in DHA Defence, Lahore"
    if bed_count:
        name = f"{bed_count} Bedroom House for sale in {society_name}, Lahore"
    else:
        name = f"House for sale in {society_name}, Lahore"

    # Main Features (dynamic columns)
    main_features = {}
    for li in soup.select("ul._3efd3392 li"):
        span = li.find("span", class_="_9121cbf9")
        if not span:
            continue
        full_text = span.get_text(separator=" ", strip=True)
        if ":" in full_text:
            k, v = full_text.split(":", 1)
            k, v = k.strip(), v.strip()
            if k:
                main_features[k] = v

    all_feature_keys.update(main_features.keys())

    floor_number = main_features.get("Floor") or main_features.get("Floor Number")
    total_floors = main_features.get("Floors in Building")
    built_year   = main_features.get("Built in year")

    # Rooms section (optional)
    rooms_section = None
    for li in soup.find_all("li", class_="_51519f00"):
        header = li.find("div", class_="_359e9f89")
        if header and header.get_text(strip=True) == "Rooms":
            rooms_section = li
            break

    room_details = {}
    other_rooms = []
    if rooms_section:
        for rli in rooms_section.find_all("li", class_="_59261156"):
            txt = rli.get_text(separator=" ", strip=True)
            if ":" in txt:
                k, v = txt.split(":", 1)
                room_details[k.strip()] = v.strip()
            else:
                other_rooms.append(txt)

    # property id (de-dupe)
    m = re.search(r"-(\d+)-\d+-\d+\.html$", prop_url)
    property_id = m.group(1) if m else "Not found"

    row = {
        "Property ID": property_id,
        "Society": society_name,
        "Society Link": society_url,      # ✅ whole link in one cell
        "Name": name,                     # ✅ naming format
        "Page Title": page_title,
        "Price": price,
        "Marla": marla_value,
        "Area (sqft)": area_sqft,
        "Bedrooms": bed_count,
        "Baths": bath_count,
        "Floor Number": floor_number,
        "Total Floors": total_floors,
        "Built Year": built_year,
        "Rooms": room_details,
        "Other Rooms": ", ".join(other_rooms),
        "Link": prop_url
    }

    # add all dynamic features as columns
    for k, v in main_features.items():
        row[k] = v

    return row

# -----------------------
# 4) Auto paginate society pages until end
# -----------------------
def scrape_society_all_pages(society_name: str, society_url: str):
    page = 1
    last_fingerprint = None

    while True:
        page_url = make_page_url(society_url, page)
        print(f"   Page {page}: {page_url}")

        try:
            links = scrape_property_links(page_url)
        except Exception as e:
            print(f"   Error listing page: {e}")
            break

        if not links:
            print("   ✅ No listings on this page. Stop society.")
            break

        fingerprint = tuple(links[:25])
        if fingerprint == last_fingerprint:
            print("   ✅ Repeated listings (end reached). Stop society.")
            break
        last_fingerprint = fingerprint

        for idx, prop_url in enumerate(links, 1):
            mid = re.search(r"-(\d+)-\d+-\d+\.html$", prop_url)
            pid = mid.group(1) if mid else None
            if pid and pid in seen_property_ids:
                continue

            try:
                row = scrape_property_details(prop_url, society_name, society_url)
                data.append(row)
                if pid:
                    seen_property_ids.add(pid)
            except Exception as e:
                print(f"     Error property: {prop_url} -> {e}")

            # if idx % 10 == 0:
            #     time.sleep(0.8)

        page += 1
        # time.sleep(random.uniform(0.4, 1.0))

# -----------------------
# RUN (Houses)
# -----------------------
master_url = "https://www.zameen.com/Houses_Property/Lahore-1-1.html"

societies = extract_societies(master_url)
print(f"Total societies found: {len(societies)}")

for i, s in enumerate(societies, 1):
    print(f"\n[{i}/{len(societies)}] Society: {s['society_name']} ({s['society_count']})")
    scrape_society_all_pages(s["society_name"], s["society_url"])

# -----------------------
# Final DataFrame
# -----------------------
df = pd.DataFrame(data)

# ensure all feature columns exist
for k in all_feature_keys:
    if k not in df.columns:
        df[k] = None

print("\n✅ DONE")
print("Rows:", len(df), "Cols:", len(df.columns))
print(df.head())

# df.to_csv("lahore_houses.csv", index=False)


Total societies found: 80

[1/80] Society: DHA Defence (7583)
   Page 1: https://www.zameen.com/Houses_Property/Lahore_DHA_Defence-9-1.html
   Page 2: https://www.zameen.com/Houses_Property/Lahore_DHA_Defence-9-2.html
   Page 3: https://www.zameen.com/Houses_Property/Lahore_DHA_Defence-9-3.html
   Page 4: https://www.zameen.com/Houses_Property/Lahore_DHA_Defence-9-4.html
   Page 5: https://www.zameen.com/Houses_Property/Lahore_DHA_Defence-9-5.html
   Page 6: https://www.zameen.com/Houses_Property/Lahore_DHA_Defence-9-6.html
   Page 7: https://www.zameen.com/Houses_Property/Lahore_DHA_Defence-9-7.html
   Page 8: https://www.zameen.com/Houses_Property/Lahore_DHA_Defence-9-8.html
   Page 9: https://www.zameen.com/Houses_Property/Lahore_DHA_Defence-9-9.html
   Page 10: https://www.zameen.com/Houses_Property/Lahore_DHA_Defence-9-10.html
   Page 11: https://www.zameen.com/Houses_Property/Lahore_DHA_Defence-9-11.html
   Page 12: https://www.zameen.com/Houses_Property/Lahore_DHA_Defence-9-12.h

In [11]:
df.to_csv("lahore_houses.csv", index=False)

In [5]:
response=requests.get("https://www.zameen.com/Property/gulberg_gulberg_3_exclusive_3_bed_gulberg_apartment_1900_sqft_fully_furnished_private_balcony_30_booking-53221997-3824-1.html")

In [6]:
soup = BeautifulSoup(response.content, 'html.parser')

# Step 3: Print the whole HTML
# .prettify() formats the HTML for easier reading
print(soup.prettify())

<!DOCTYPE html>
<html dir="ltr" lang="en">
 <head>
  <meta charset="utf-8"/>
  <meta content="width=device-width, initial-scale=1.0, user-scalable=0" name="viewport"/>
  <link href="http://5BFU7FLVAD-dsn.algolia.net" rel="dns-prefetch"/>
  <link href="https://www.googletagmanager.com" rel="dns-prefetch"/>
  <link href="https://www.google-analytics.com" rel="dns-prefetch"/>
  <link href="https://www.googletagservices.com" rel="dns-prefetch"/>
  <link href="https://www.googleadservices.com" rel="dns-prefetch"/>
  <link href="https://images.zameen.com" rel="dns-prefetch"/>
  <link href="https://maps.googleapis.com/" rel="dns-prefetch"/>
  <link href="https://www.zameen.com/" rel="dns-prefetch"/>
  <link href="https://securepubads.g.doubleclick.net/" rel="dns-prefetch"/>
  <link href="https://widget.eu.criteo.com/" rel="dns-prefetch"/>
  <link href="/assets/apple-touch-icon.1dbbcc9e3f454ad14d4fd0f5a25d37a3.png" rel="apple-touch-icon" sizes="180x180"/>
  <link href="/assets/favicon-16x16.8c

In [8]:
soup.find(class_="_83bb17d1")

<div class="_83bb17d1"><h3 class="_321aa0bf">Details</h3><ul aria-label="Property details" class="_3dc8d08d" style="columns:2"><li><span class="ed0db22a">Type</span><span aria-label="Type" class="_2fdf7fc5">Flat</span></li><li><span class="ed0db22a">Price</span><span aria-label="Price" class="_2fdf7fc5"><div class="_2923a568">PKR<span class="_9e0180f9"></span>5.39 Crore</div></span></li><li><span class="ed0db22a">Bath(s)</span><span aria-label="Baths" class="_2fdf7fc5">3</span></li><li><span class="ed0db22a">Area</span><span aria-label="Area" class="_2fdf7fc5"><span>8.4 Marla</span></span></li><li><span class="ed0db22a">Purpose</span><span aria-label="Purpose" class="_2fdf7fc5">For Sale</span></li><li><span class="ed0db22a">Bedroom(s)</span><span aria-label="Beds" class="_2fdf7fc5">3</span></li><li><span class="ed0db22a">Added</span><span aria-label="Creation date" class="_2fdf7fc5">4 weeks ago</span></li><li><span class="ed0db22a">Location</span><span aria-label="Location" class="_2fd

## v2

In [2]:
import re
import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE = "https://www.zameen.com"

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9"
})

data = []
all_feature_keys = set()
seen_property_ids = set()

# -----------------------
# Overview label helpers
# -----------------------
def is_area_li(tag):
    if tag.name != "li":
        return False
    label = tag.find("span", class_="ed0db22a")
    return label and label.get_text(strip=True) == "Area"

def is_bedroom_li(tag):
    if tag.name != "li":
        return False
    label = tag.find("span", class_="ed0db22a")
    return label and label.get_text(strip=True) in ["Bedroom(s)", "Beds"]

def is_bath_li(tag):
    if tag.name != "li":
        return False
    label = tag.find("span", class_="ed0db22a")
    return label and label.get_text(strip=True) in ["Bath(s)", "Baths"]

def clean(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip()

def get_soup(url, timeout=25):
    r = session.get(url, timeout=timeout)
    r.raise_for_status()
    return BeautifulSoup(r.content, "html.parser")

def make_page_url(seed_url: str, page: int) -> str:
    # ...-1.html -> ...-2.html -> ...
    if re.search(r"-\d+\.html$", seed_url):
        return re.sub(r"-\d+\.html$", f"-{page}.html", seed_url)
    return seed_url

# -----------------------
# 1) Extract societies from master page
# -----------------------
def extract_societies(master_url: str):
    soup = get_soup(master_url)
    wrapper = soup.find(class_="_5d76b098")
    if not wrapper:
        return []

    items = wrapper.find_all("li", class_="_6d61972c")
    societies = []

    for li in items:
        a = li.find("a", class_="_6de4d4f7", href=True)
        if not a:
            continue

        href = urljoin(BASE, a["href"])
        full_text = a.get_text(" ", strip=True)

        name = re.sub(r"\(\s*[\d,]+\s*\)\s*$", "", full_text).strip()
        m = re.search(r"\(\s*([\d,]+)\s*\)\s*$", full_text)
        count = int(m.group(1).replace(",", "")) if m else None

        societies.append({
            "society_name": name,
            "society_count": count,
            "society_url": href
        })

    return societies

# -----------------------
# 2) Listing page -> property links
#    ✅ handles 404 gracefully (end pages)
# -----------------------
def scrape_property_links(listing_url: str):
    r = session.get(listing_url, timeout=25)
    if r.status_code == 404:
        return []
    r.raise_for_status()

    soup = BeautifulSoup(r.content, "html.parser")

    links = []
    ul = soup.find("ul", class_="e20beb46")
    if ul:
        for a in ul.find_all("a", class_="d870ae17", href=True):
            links.append(urljoin(BASE, a["href"]))

    return list(dict.fromkeys(links))

# -----------------------
# NEW: parse ALL feature sections (Main Features, Rooms, Nearby, Other Facilities, etc.)
# -----------------------
def parse_all_sections(soup: BeautifulSoup):
    """
    Returns dict like:
      {
        "Main Features": ["Built in year: 2026", ...],
        "Rooms": ["Bedrooms: 3", "Dining Room", ...],
        "Nearby Locations and Other Facilities": [...],
        ...
      }
    """
    out = {}
    ul = soup.find("ul", class_="_49fc0232")
    if not ul:
        return out

    for sec in ul.find_all("li", class_="_51519f00"):
        header = sec.find("div", class_="_359e9f89")
        title = clean(header.get_text(" ", strip=True)) if header else None
        if not title:
            continue

        items = []
        items_ul = sec.find("ul", class_="_3efd3392")
        if items_ul:
            for li in items_ul.find_all("li", class_="_59261156"):
                span = li.find("span", class_="_9121cbf9")
                if span:
                    txt = clean(span.get_text(" ", strip=True))
                    if txt:
                        items.append(txt)

        if items:
            out[title] = items

    return out

def parse_rooms(section_map: dict):
    rooms_items = section_map.get("Rooms") or []
    room_details = {}
    other_rooms = []
    for it in rooms_items:
        if ":" in it:
            k, v = it.split(":", 1)
            room_details[clean(k)] = clean(v)
        else:
            other_rooms.append(it)
    return room_details, other_rooms

# -----------------------
# 3) Property page -> details (UPDATED with Description + Features + Nearby)
# -----------------------
def scrape_property_details(prop_url: str, society_name: str, society_url: str):
    global all_feature_keys

    soup = get_soup(prop_url)

    title_tag = soup.find(class_="cd230541")
    price_tag = soup.find(class_="_2923a568")

    area_li = soup.find(is_area_li)
    bed_li  = soup.find(is_bedroom_li)
    bath_li = soup.find(is_bath_li)

    bed_count = bath_count = marla_value = area_sqft = None

    if bed_li:
        s = bed_li.find("span", class_="_2fdf7fc5")
        bed_count = s.get_text(strip=True) if s else None

    if bath_li:
        s = bath_li.find("span", class_="_2fdf7fc5")
        bath_count = s.get_text(strip=True) if s else None

    if area_li:
        s = area_li.find("span", class_="_2fdf7fc5")
        if s:
            area_text = s.get_text(strip=True)
            try:
                marla_value = float(area_text.split()[0])
                area_sqft = marla_value * 225
            except:
                pass

    # price
    price_raw = price_tag.get_text(strip=True) if price_tag else ""
    price = "PKR " + price_raw[3:].strip() if (price_raw.startswith("PKR") and not price_raw.startswith("PKR ")) else price_raw

    # page title / address
    page_title = title_tag.get_text(strip=True) if title_tag else "Not listed"
    address = page_title

    # naming (keep your house naming)
    if bed_count:
        name = f"{bed_count} Bedroom House for sale in {society_name}, Lahore"
    else:
        name = f"House for sale in {society_name}, Lahore"

    # ✅ NEW: Description
    desc_tag = soup.find(class_="_3547dac9")
    description = clean(desc_tag.get_text(" ", strip=True)) if desc_tag else None

    # ✅ NEW: ALL sections (Main Features, Rooms, Nearby, Other Facilities...)
    section_map = parse_all_sections(soup)

    # ✅ Rooms same logic
    room_details, other_rooms = parse_rooms(section_map)

    # ✅ NEW: Nearby Locations and Other Facilities (separate column)
    nearby_key = "Nearby Locations and Other Facilities"
    nearby_locations = ", ".join(section_map.get(nearby_key, [])) if section_map.get(nearby_key) else None

    # ✅ NEW: Merge ALL features from ALL sections except Rooms
    merged_features = []
    for sec_title, items in section_map.items():
        if sec_title == "Rooms":
            continue
        merged_features.extend(items)

    features_str = " | ".join(merged_features) if merged_features else None

    # ✅ Dynamic features dict (key:value + boolean) from merged_features
    main_features = {}
    for t in merged_features:
        if ":" in t:
            k, v = t.split(":", 1)
            k, v = clean(k), clean(v)
            if k:
                main_features[k] = v
        else:
            main_features[clean(t)] = True

    all_feature_keys.update(main_features.keys())

    # mapped columns
    floor_number = main_features.get("Floor") or main_features.get("Floor Number")
    total_floors = main_features.get("Floors in Building")
    built_year   = main_features.get("Built in year")

    # property id
    m = re.search(r"-(\d+)-\d+-\d+\.html$", prop_url)
    property_id = m.group(1) if m else "Not found"

    row = {
        "Property ID": property_id,
        "Society": society_name,
        "Society Link": society_url,
        "Name": name,
        "Page Title": page_title,
        "Price": price,
        "Marla": marla_value,
        "Area (sqft)": area_sqft,
        "Bedrooms": bed_count,
        "Baths": bath_count,
        "Floor Number": floor_number,
        "Total Floors": total_floors,
        "Built Year": built_year,
        "Address": address,

        # ✅ NEW columns
        "Description": description,
        "Features": features_str,
        "Nearby Locations and Other Facilities": nearby_locations,

        # Rooms
        "Rooms": room_details,
        "Other Rooms": ", ".join(other_rooms),

        "Link": prop_url
    }

    # add dynamic features as columns
    for k, v in main_features.items():
        row[k] = v

    return row

# -----------------------
# 4) Auto paginate society pages until end
# -----------------------
def scrape_society_all_pages(society_name: str, society_url: str):
    page = 1
    last_fingerprint = None

    while True:
        page_url = make_page_url(society_url, page)
        print(f"   Page {page}: {page_url}")

        try:
            links = scrape_property_links(page_url)
        except Exception as e:
            print(f"   Error listing page: {e}")
            break

        if not links:
            print("   ✅ No listings on this page. Stop society.")
            break

        fingerprint = tuple(links[:25])
        if fingerprint == last_fingerprint:
            print("   ✅ Repeated listings (end reached). Stop society.")
            break
        last_fingerprint = fingerprint

        for prop_url in links:
            mid = re.search(r"-(\d+)-\d+-\d+\.html$", prop_url)
            pid = mid.group(1) if mid else None
            if pid and pid in seen_property_ids:
                continue

            try:
                row = scrape_property_details(prop_url, society_name, society_url)
                data.append(row)
                if pid:
                    seen_property_ids.add(pid)
            except Exception as e:
                print(f"     Error property: {prop_url} -> {e}")

        page += 1

# -----------------------
# RUN (Houses or Flats)
# -----------------------
master_url = "https://www.zameen.com/Houses_Property/Lahore-1-1.html"
# If you want flats master, change to:
# master_url = "https://www.zameen.com/Flats_Apartments/Lahore-1-1.html"

societies = extract_societies(master_url)
print(f"Total societies found: {len(societies)}")

for i, s in enumerate(societies, 1):
    print(f"\n[{i}/{len(societies)}] Society: {s['society_name']} ({s['society_count']})")
    scrape_society_all_pages(s["society_name"], s["society_url"])

# -----------------------
# Final DataFrame
# -----------------------
df = pd.DataFrame(data)

# ensure all dynamic feature columns exist
for k in all_feature_keys:
    if k not in df.columns:
        df[k] = None

print("\n✅ DONE")
print("Rows:", len(df), "Cols:", len(df.columns))
print(df.head())

df.to_csv("lahore_data_v2.csv", index=False, encoding="utf-8-sig")

Total societies found: 80

[1/80] Society: DHA Defence (7567)
   Page 1: https://www.zameen.com/Houses_Property/Lahore_DHA_Defence-9-1.html
   Page 2: https://www.zameen.com/Houses_Property/Lahore_DHA_Defence-9-2.html
   Page 3: https://www.zameen.com/Houses_Property/Lahore_DHA_Defence-9-3.html
   Page 4: https://www.zameen.com/Houses_Property/Lahore_DHA_Defence-9-4.html
   Page 5: https://www.zameen.com/Houses_Property/Lahore_DHA_Defence-9-5.html
   Page 6: https://www.zameen.com/Houses_Property/Lahore_DHA_Defence-9-6.html
   Page 7: https://www.zameen.com/Houses_Property/Lahore_DHA_Defence-9-7.html
   Page 8: https://www.zameen.com/Houses_Property/Lahore_DHA_Defence-9-8.html
   Page 9: https://www.zameen.com/Houses_Property/Lahore_DHA_Defence-9-9.html
   Page 10: https://www.zameen.com/Houses_Property/Lahore_DHA_Defence-9-10.html
   Page 11: https://www.zameen.com/Houses_Property/Lahore_DHA_Defence-9-11.html
   Page 12: https://www.zameen.com/Houses_Property/Lahore_DHA_Defence-9-12.h

KeyboardInterrupt: 

In [ ]:
import re
import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE = "https://www.zameen.com"

# Create session with retry logic
session = requests.Session()
retry_strategy = Retry(
    total=3,
    backoff_factor=2,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["HEAD", "GET", "OPTIONS"]
)
adapter = HTTPAdapter(max_retries=retry_strategy)
session.mount("https://", adapter)
session.mount("http://", adapter)

session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.zameen.com/",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
})

data = []
all_feature_keys = set()
seen_property_ids = set()

# Error tracking
failed_pages = []
failed_properties = []
first_fetch_printed = False

# Request counter for rate limiting
request_count = 0
last_request_time = time.time()

# -----------------------
# Rate limiting helper
# -----------------------
def rate_limit():
    """Implement rate limiting - max 1 request per 1-2 seconds"""
    global request_count, last_request_time
    
    current_time = time.time()
    time_since_last = current_time - last_request_time
    
    # Random delay between 1-2 seconds
    min_delay = 1.0
    max_delay = 2.0
    delay = random.uniform(min_delay, max_delay)
    
    if time_since_last < delay:
        time.sleep(delay - time_since_last)
    
    last_request_time = time.time()
    request_count += 1
    
    # Every 50 requests, take a longer break
    if request_count % 50 == 0:
        print(f"\n   💤 Taking 10 second break after {request_count} requests...\n")
        time.sleep(10)

# -----------------------
# Overview label helpers
# -----------------------
def is_area_li(tag):
    if tag.name != "li":
        return False
    label = tag.find("span", class_="ed0db22a")
    return label and label.get_text(strip=True) == "Area"

def is_bedroom_li(tag):
    if tag.name != "li":
        return False
    label = tag.find("span", class_="ed0db22a")
    return label and label.get_text(strip=True) in ["Bedroom(s)", "Beds"]

def is_bath_li(tag):
    if tag.name != "li":
        return False
    label = tag.find("span", class_="ed0db22a")
    return label and label.get_text(strip=True) in ["Bath(s)", "Baths"]

def clean(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip()

def get_soup(url, timeout=30, max_retries=3):
    """Get soup with error handling and retry logic"""
    for attempt in range(max_retries):
        try:
            rate_limit()  # Apply rate limiting
            r = session.get(url, timeout=timeout)
            r.raise_for_status()
            return BeautifulSoup(r.content, "html.parser")
        except requests.exceptions.HTTPError as e:
            if e.response.status_code == 404:
                return None  # Silently handle 404
            else:
                if attempt < max_retries - 1:
                    wait_time = (attempt + 1) * 5
                    print(f"  ⚠️  HTTP {e.response.status_code}, retrying in {wait_time}s... (attempt {attempt + 1}/{max_retries})")
                    time.sleep(wait_time)
                else:
                    print(f"  ⚠️  HTTP Error {e.response.status_code} after {max_retries} attempts")
                    return None
        except requests.exceptions.Timeout:
            if attempt < max_retries - 1:
                wait_time = (attempt + 1) * 5
                print(f"  ⚠️  Timeout, retrying in {wait_time}s... (attempt {attempt + 1}/{max_retries})")
                time.sleep(wait_time)
            else:
                print(f"  ⚠️  Timeout after {max_retries} attempts")
                return None
        except (requests.exceptions.ConnectionError, requests.exceptions.ChunkedEncodingError) as e:
            if attempt < max_retries - 1:
                wait_time = (attempt + 1) * 10  # Longer wait for connection errors
                print(f"  ⚠️  Connection error, retrying in {wait_time}s... (attempt {attempt + 1}/{max_retries})")
                time.sleep(wait_time)
            else:
                print(f"  ⚠️  Connection error after {max_retries} attempts")
                return None
        except requests.exceptions.RequestException as e:
            if attempt < max_retries - 1:
                wait_time = (attempt + 1) * 5
                print(f"  ⚠️  Request error, retrying in {wait_time}s...")
                time.sleep(wait_time)
            else:
                print(f"  ⚠️  Request error: {str(e)[:100]}")
                return None
        except Exception as e:
            print(f"  ⚠️  Unexpected error: {str(e)[:100]}")
            return None
    
    return None

def make_page_url(seed_url: str, page: int) -> str:
    # ...-1.html -> ...-2.html -> ...
    if re.search(r"-\d+\.html$", seed_url):
        return re.sub(r"-\d+\.html$", f"-{page}.html", seed_url)
    return seed_url

# -----------------------
# 1) Extract societies from master page
# -----------------------
def extract_societies(master_url: str):
    """Extract all societies with error handling"""
    try:
        print(f"Fetching master page: {master_url}")
        soup = get_soup(master_url)
        if soup is None:
            print("⚠️  Failed to fetch master page")
            return []
        
        wrapper = soup.find(class_="_5d76b098")
        if not wrapper:
            print("⚠️  Could not find societies wrapper")
            return []

        items = wrapper.find_all("li", class_="_6d61972c")
        if not items:
            print("⚠️  No society items found")
            return []
            
        societies = []

        for li in items:
            a = li.find("a", class_="_6de4d4f7", href=True)
            if not a:
                continue

            href = urljoin(BASE, a["href"])
            full_text = a.get_text(" ", strip=True)

            name = re.sub(r"\(\s*[\d,]+\s*\)\s*$", "", full_text).strip()
            m = re.search(r"\(\s*([\d,]+)\s*\)\s*$", full_text)
            count = int(m.group(1).replace(",", "")) if m else None

            societies.append({
                "society_name": name,
                "society_count": count,
                "society_url": href
            })

        return societies
    except Exception as e:
        print(f"⚠️  Error extracting societies: {str(e)}")
        return []

# -----------------------
# 2) Listing page -> property links
# -----------------------
def scrape_property_links(listing_url: str):
    """Scrape property links with error handling"""
    try:
        soup = get_soup(listing_url)
        if soup is None:
            return []

        links = []
        ul = soup.find("ul", class_="e20beb46")
        if ul:
            for a in ul.find_all("a", class_="d870ae17", href=True):
                links.append(urljoin(BASE, a["href"]))

        return list(dict.fromkeys(links))
    except Exception as e:
        print(f"  ⚠️  Error extracting links: {str(e)[:100]}")
        return []

# -----------------------
# Parse ALL feature sections
# -----------------------
def parse_all_sections(soup: BeautifulSoup):
    out = {}
    try:
        ul = soup.find("ul", class_="_49fc0232")
        if not ul:
            return out

        for sec in ul.find_all("li", class_="_51519f00"):
            header = sec.find("div", class_="_359e9f89")
            title = clean(header.get_text(" ", strip=True)) if header else None
            if not title:
                continue

            items = []
            items_ul = sec.find("ul", class_="_3efd3392")
            if items_ul:
                for li in items_ul.find_all("li", class_="_59261156"):
                    span = li.find("span", class_="_9121cbf9")
                    if span:
                        txt = clean(span.get_text(" ", strip=True))
                        if txt:
                            items.append(txt)

            if items:
                out[title] = items
    except Exception as e:
        pass  # Silent fail for sections parsing
    
    return out

def parse_rooms(section_map: dict):
    try:
        rooms_items = section_map.get("Rooms") or []
        room_details = {}
        other_rooms = []
        for it in rooms_items:
            if ":" in it:
                k, v = it.split(":", 1)
                room_details[clean(k)] = clean(v)
            else:
                other_rooms.append(it)
        return room_details, other_rooms
    except Exception:
        return {}, []

# -----------------------
# 3) Property page -> details
# -----------------------
def scrape_property_details(prop_url: str, society_name: str, society_url: str):
    global all_feature_keys, first_fetch_printed

    try:
        soup = get_soup(prop_url)
        if soup is None:
            failed_properties.append({"url": prop_url, "error": "Failed to fetch"})
            return None

        title_tag = soup.find(class_="cd230541")
        price_tag = soup.find(class_="_2923a568")

        area_li = soup.find(is_area_li)
        bed_li  = soup.find(is_bedroom_li)
        bath_li = soup.find(is_bath_li)

        bed_count = bath_count = marla_value = area_sqft = None

        if bed_li:
            s = bed_li.find("span", class_="_2fdf7fc5")
            bed_count = clean(s.get_text(strip=True)) if s else None

        if bath_li:
            s = bath_li.find("span", class_="_2fdf7fc5")
            bath_count = clean(s.get_text(strip=True)) if s else None

        if area_li:
            s = area_li.find("span", class_="_2fdf7fc5")
            if s:
                area_text = clean(s.get_text(strip=True))
                try:
                    m = re.search(r"([\d.]+)", area_text)
                    if m:
                        marla_value = float(m.group(1))
                        area_sqft = marla_value * 225
                except:
                    pass

        # price
        price_raw = clean(price_tag.get_text(strip=True)) if price_tag else ""
        price = "PKR " + price_raw[3:].strip() if (price_raw.startswith("PKR") and not price_raw.startswith("PKR ")) else price_raw

        # page title / address
        page_title = clean(title_tag.get_text(strip=True)) if title_tag else "Not listed"
        address = page_title

        # naming
        if bed_count:
            name = f"{bed_count} Bedroom House for sale in {society_name}, Lahore"
        else:
            name = f"House for sale in {society_name}, Lahore"

        # Description
        desc_tag = soup.find(class_="_3547dac9")
        description = clean(desc_tag.get_text(" ", strip=True)) if desc_tag else None

        # ALL sections
        section_map = parse_all_sections(soup)

        # Rooms
        room_details, other_rooms = parse_rooms(section_map)

        # Nearby Locations
        nearby_key = "Nearby Locations and Other Facilities"
        nearby_locations = ", ".join(section_map.get(nearby_key, [])) if section_map.get(nearby_key) else None

        # Merge ALL features
        merged_features = []
        for sec_title, items in section_map.items():
            if sec_title == "Rooms":
                continue
            merged_features.extend(items)

        features_str = " | ".join(merged_features) if merged_features else None

        # Dynamic features dict
        main_features = {}
        for t in merged_features:
            if ":" in t:
                k, v = t.split(":", 1)
                k, v = clean(k), clean(v)
                if k:
                    main_features[k] = v
            else:
                main_features[clean(t)] = True

        all_feature_keys.update(main_features.keys())

        # mapped columns
        floor_number = main_features.get("Floor") or main_features.get("Floor Number")
        total_floors = main_features.get("Floors in Building")
        built_year   = main_features.get("Built in year")

        # property id
        m = re.search(r"-(\d+)-\d+-\d+\.html$", prop_url)
        property_id = m.group(1) if m else "Not found"

        row = {
            "Property ID": property_id,
            "Society": society_name,
            "Society Link": society_url,
            "Name": name,
            "Page Title": page_title,
            "Price": price,
            "Marla": marla_value,
            "Area (sqft)": area_sqft,
            "Bedrooms": bed_count,
            "Baths": bath_count,
            "Floor Number": floor_number,
            "Total Floors": total_floors,
            "Built Year": built_year,
            "Address": address,
            "Description": description,
            "Features": features_str,
            "Nearby Locations and Other Facilities": nearby_locations,
            "Rooms": room_details,
            "Other Rooms": ", ".join(other_rooms),
            "Link": prop_url
        }

        # add dynamic features as columns
        for k, v in main_features.items():
            row[k] = v

        # Add section columns
        for sec_title, items in section_map.items():
            row[f"Section - {sec_title}"] = ", ".join(items)

        # ✅ PRINT FIRST FETCH DETAILS
        if not first_fetch_printed:
            print("\n" + "=" * 80)
            print("🔍 FIRST PROPERTY DETAILS")
            print("=" * 80)
            print(f"URL: {prop_url}")
            print(f"Society: {society_name}")
            print(f"Price: {price}")
            print(f"Bedrooms: {bed_count} | Baths: {bath_count}")
            print(f"Area: {marla_value} marla ({area_sqft} sqft)")
            print(f"Sections found: {len(section_map)}")
            print(f"Dynamic features: {len(main_features)}")
            print(f"Total columns: {len(row)}")
            print("=" * 80 + "\n")
            first_fetch_printed = True

        return row

    except Exception as e:
        error_msg = f"{type(e).__name__}: {str(e)}"
        failed_properties.append({"url": prop_url, "error": error_msg})
        return None

# -----------------------
# 4) Auto paginate society pages
# -----------------------
def scrape_society_all_pages(society_name: str, society_url: str):
    page = 1
    last_fingerprint = None
    total_scraped = 0
    consecutive_failures = 0

    while True:
        page_url = make_page_url(society_url, page)
        print(f"   Page {page}: Fetching...")

        links = scrape_property_links(page_url)

        if not links:
            consecutive_failures += 1
            if consecutive_failures >= 2:
                print("   ✅ No listings for 2 consecutive pages. Stopping.")
                break
            else:
                print("   ⚠️  No listings, trying next page...")
                page += 1
                continue
        else:
            consecutive_failures = 0

        # Check for repeated listings
        fingerprint = tuple(links[:25])
        if fingerprint == last_fingerprint:
            print("   ✅ Repeated listings. Stopping.")
            break
        last_fingerprint = fingerprint

        print(f"   Found {len(links)} properties on page {page}")

        for idx, prop_url in enumerate(links, 1):
            mid = re.search(r"-(\d+)-\d+-\d+\.html$", prop_url)
            pid = mid.group(1) if mid else None
            if pid and pid in seen_property_ids:
                continue

            row = scrape_property_details(prop_url, society_name, society_url)
            if row:
                data.append(row)
                total_scraped += 1
                if pid:
                    seen_property_ids.add(pid)
            
            # Progress indicator
            if idx % 10 == 0:
                print(f"   Scraped {idx}/{len(links)} from page {page}")

        print(f"   ✅ Page {page} complete. Total from society: {total_scraped}")
        page += 1

# -----------------------
# RUN
# -----------------------
print("=" * 80)
print("ZAMEEN.COM PROPERTY SCRAPER")
print("=" * 80)

# Choose one:
master_url = "https://www.zameen.com/Houses_Property/Lahore-1-1.html"  # For Houses
# master_url = "https://www.zameen.com/Flats_Apartments/Lahore-1-1.html"  # For Flats

print(f"\nMaster URL: {master_url}")
print("⚠️  Rate limiting: 1-2 seconds between requests")
print("⚠️  10 second break every 50 requests\n")

societies = extract_societies(master_url)

if not societies:
    print("\n⚠️  No societies found. Exiting.")
    exit()

print(f"✅ Found {len(societies)} societies\n")

for i, s in enumerate(societies, 1):
    print(f"\n{'=' * 80}")
    print(f"[{i}/{len(societies)}] {s['society_name']} ({s['society_count']} properties)")
    print("=" * 80)
    scrape_society_all_pages(s["society_name"], s["society_url"])
    
    # Save progress after each society
    if len(data) > 0 and i % 5 == 0:
        df_temp = pd.DataFrame(data)
        for k in all_feature_keys:
            if k not in df_temp.columns:
                df_temp[k] = None
        df_temp.to_csv(f"lahore_properties_backup_{i}.csv", index=False, encoding="utf-8-sig")
        print(f"\n💾 Progress saved: {len(data)} properties\n")

# -----------------------
# Final DataFrame
# -----------------------
print(f"\n{'=' * 80}")
print("SCRAPING COMPLETE")
print("=" * 80)
print(f"✅ Properties collected: {len(data)}")
print(f"❌ Failed properties: {len(failed_properties)}")
print(f"📊 Total requests made: {request_count}")

if len(data) == 0:
    print("\n⚠️  No data collected!")
    exit()

df = pd.DataFrame(data)

# ensure all dynamic feature columns exist
for k in all_feature_keys:
    if k not in df.columns:
        df[k] = None

print(f"\nDataFrame: {len(df)} rows, {len(df.columns)} columns")

output_file = "lahore_properties_final.csv"
df.to_csv(
          
          , index=False, encoding="utf-8-sig")
print(f"\n✅ Saved to {output_file}")

# Save errors if any
if failed_properties:
    import json
    with open("scraping_errors.json", "w") as f:
        json.dump({"failed_properties": failed_properties}, f, indent=2)
    print(f"⚠️  Error log: scraping_errors.json ({len(failed_properties)} failures)")

ZAMEEN.COM PROPERTY SCRAPER

Master URL: https://www.zameen.com/Houses_Property/Lahore-1-1.html
⚠️  Rate limiting: 1-2 seconds between requests
⚠️  10 second break every 50 requests

Fetching master page: https://www.zameen.com/Houses_Property/Lahore-1-1.html
✅ Found 80 societies


[1/80] DHA Defence (7528 properties)
   Page 1: Fetching...
   Found 27 properties on page 1

🔍 FIRST PROPERTY DETAILS
URL: https://www.zameen.com/Property/dha_phase_5_dha_phase_5_-_block_b_34_marla_slightly_used_spanish_design_7-bed_room_facing_park_bungalow_for_sale_at_prime_location_of_dha_lahore_near_to_park_masjid_commercial_market-53483264-1599-1.html
Society: DHA Defence
Price: PKR 18.5 Crore
Bedrooms: 7 | Baths: 7
Area: 1.7 marla (382.5 sqft)
Sections found: 7
Dynamic features: 32
Total columns: 59

   Scraped 10/27 from page 1
   Scraped 20/27 from page 1
   ✅ Page 1 complete. Total from society: 27
   Page 2: Fetching...
   Found 27 properties on page 2
   Scraped 10/27 from page 2

   💤 Taking 10 